# 01 · Parameter Analysis

Load classified documents from the Veritas API and visualise the distributions of extracted
experimental parameters (excess heat, COP, loading ratio, etc.).

> **[MOCK]** Data loading cells below pull from the mock API.  
> Replace `API_BASE` with the real endpoint and remove the mock-data fallback once the
> extraction pipeline returns real values (requires physicist session for field mappings).

In [ ]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
API_BASE = os.getenv('VERITAS_API_BASE', 'http://localhost:5000')

In [ ]:
# [MOCK] Replace corpus_id with a real corpus ID from the platform
CORPUS_ID = 'demo-lenr-corpus'

resp = requests.get(f'{API_BASE}/api/corpora/{CORPUS_ID}/documents')
if resp.ok:
    documents = resp.json()
else:
    # [MOCK] Fallback mock data — remove once real API is connected
    documents = [
        {'document_id': 'doc-001', 'title': 'Mock Experiment A', 'status': 'classified',
         'extracted_fields': {'excess_heat_watts': 8.5, 'cop': 1.4, 'loading_ratio': 0.89}},
        {'document_id': 'doc-002', 'title': 'Mock Experiment B', 'status': 'classified',
         'extracted_fields': {'excess_heat_watts': 12.1, 'cop': 1.7, 'loading_ratio': 0.91}},
        {'document_id': 'doc-003', 'title': 'Mock Experiment C', 'status': 'classified',
         'extracted_fields': {'excess_heat_watts': 5.0, 'cop': 1.2, 'loading_ratio': 0.85}},
    ]
    print('[MOCK] Using fallback data')

print(f'Loaded {len(documents)} documents')

In [ ]:
# Flatten extracted_fields into a dataframe
rows = []
for doc in documents:
    row = {'document_id': doc['document_id'], 'title': doc.get('title', '')}
    row.update(doc.get('extracted_fields', {}))
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

In [ ]:
# [MOCK] Parameter columns — update to match real extracted field names from the LENR ontology
PARAM_COLS = ['excess_heat_watts', 'cop', 'loading_ratio']
available = [c for c in PARAM_COLS if c in df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
if len(available) == 1:
    axes = [axes]

for ax, col in zip(axes, available):
    vals = df[col].dropna()
    sns.histplot(vals, kde=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('value')

plt.suptitle('Parameter Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('parameter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
if available:
    display(df[available].describe())